[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/ia_p26/blob/main/clase/12_montecarlo/notebooks/aplicaciones/06_ising_metropolis.ipynb)

# Aplicación 6: El Modelo de Ising y Metropolis-Hastings

## El círculo que se cierra

En 1953, Nicholas Metropolis, Arianna Rosenbluth, Marshall Rosenbluth, Augusta Teller y Edward Teller publicaron
*"Equation of State Calculations by Fast Computing Machines"* — el paper que inventó el algoritmo de Metropolis-Hastings.

El problema que querían resolver era exactamente este: simular un sistema de partículas magnéticas para estudiar
transiciones de fase. Hoy ese problema se llama el **modelo de Ising**.

Este notebook cierra el círculo histórico: arrancamos con la historia de Los Álamos, y ahora simulamos
el problema exacto para el que se inventó el algoritmo más influyente en estadística computacional moderna.

## Por qué es importante hoy

El mismo algoritmo (Metropolis-Hastings) es la base de:
- **MCMC** en inferencia bayesiana (Notebook 04)
- **Máquinas de Boltzmann** — los antepasados de las redes neuronales profundas
- **Modelos de difusión** — la tecnología detrás de DALL-E, Stable Diffusion, y Midjourney
- **Optimización combinatoria** (recocido simulado)

Aprender a simular el modelo de Ising es aprender la mecánica de muestreo más usada en ML moderno.

In [ ]:
# Descomenta si estás en Colab
# !pip install numpy matplotlib scipy --quiet

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.colors import ListedColormap

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["font.size"] = 11

COLORS = {"blue":"#2E86AB","red":"#E94F37","green":"#27AE60","orange":"#F39C12","purple":"#8E44AD"}

rng = np.random.default_rng(42)
print("Imports listos ✓")

---
## Sección 1: El Problema — Demasiados Estados

### El modelo de Ising

El **modelo de Ising** describe un material magnético como una rejilla de $N \times N$ *espines*,
cada uno con valor $s_i \in \{+1, -1\}$ (piénsalos como mini-imanes que apuntan arriba o abajo).

Los espines vecinos interactúan: prefieren alinearse. La energía total del sistema es:

$$H(\mathbf{s}) = -J \sum_{\langle i,j \rangle} s_i s_j$$

donde la suma es sobre todos los pares de vecinos adyacentes y $J > 0$ favorece la alineación.

En equilibrio térmico a temperatura $T$, la probabilidad de cada configuración $\mathbf{s}$ sigue la
**distribución de Boltzmann**:

$$P(\mathbf{s}) = \frac{1}{Z} e^{-H(\mathbf{s}) / k_B T}$$

donde $Z = \sum_{\mathbf{s}} e^{-H(\mathbf{s})/k_B T}$ es la función de partición (una constante normalizadora).

### El problema computacional

Para una rejilla de $20 \times 20 = 400$ espines, hay $2^{400} \approx 10^{120}$ configuraciones posibles.
Enumerar todas es literalmente imposible — hay más configuraciones que átomos en el universo observable.

¿Cómo calculamos entonces el valor esperado de cualquier observable (magnetización, energía, etc.)?

**Respuesta**: Monte Carlo — pero con un truco. No podemos muestrear directamente de $P(\mathbf{s})$
porque no podemos calcular $Z$. Necesitamos un método que muestree de $P$ sin necesitar conocer $Z$.

---
## Sección 2: Metropolis-Hastings — Muestrear sin conocer Z

### La idea clave

El algoritmo de Metropolis-Hastings construye una **cadena de Markov** cuya distribución estacionaria
es exactamente $P(\mathbf{s})$. No necesitamos calcular $Z$ porque en cada paso solo necesitamos el
*ratio* de probabilidades:

$$\frac{P(\mathbf{s}')}{P(\mathbf{s})} = e^{-(H(\mathbf{s}') - H(\mathbf{s}))/k_B T} = e^{-\Delta H / k_B T}$$

La $Z$ se cancela en el cociente.

### El algoritmo para el modelo de Ising

En cada paso:
1. Elige un espín $i$ al azar
2. Calcula $\Delta H$ = cambio en energía si flip ese espín
3. Si $\Delta H \leq 0$: acepta el flip (la energía baja → siempre conveniente)
4. Si $\Delta H > 0$: acepta el flip con probabilidad $e^{-\Delta H / k_B T}$

El criterio de aceptación es `min(1, exp(-ΔH/kT))`. Con temperatura alta ($T \to \infty$), casi todo
se acepta → configuración aleatoria. Con temperatura baja ($T \to 0$), casi nada se acepta →
configuración de mínima energía (todos los espines alineados).

In [ ]:
def init_lattice(N, kind="random"):
    """Inicializa una rejilla NxN de espines ±1."""
    if kind == "random":
        return rng.choice([-1, 1], size=(N, N)).astype(np.int8)
    elif kind == "all_up":
        return np.ones((N, N), dtype=np.int8)
    else:
        return -np.ones((N, N), dtype=np.int8)

def delta_energy(lattice, i, j):
    """Cambio de energía si se flippea el espín (i,j). Condiciones de frontera periódicas."""
    N = lattice.shape[0]
    s = lattice[i, j]
    # Suma de vecinos (frontera periódica)
    neighbors = (lattice[(i+1) % N, j] + lattice[(i-1) % N, j] +
                 lattice[i, (j+1) % N] + lattice[i, (j-1) % N])
    return 2 * s * neighbors   # J=1

def metropolis_step(lattice, T):
    """Un paso del algoritmo de Metropolis: propone N^2 flips (un barrido)."""
    N = lattice.shape[0]
    for _ in range(N * N):
        i, j = rng.integers(0, N, size=2)
        dE = delta_energy(lattice, i, j)
        if dE <= 0 or rng.random() < np.exp(-dE / T):
            lattice[i, j] *= -1
    return lattice

def magnetization(lattice):
    """Magnetización promedio por espín: M = |sum(s)| / N^2"""
    return np.abs(lattice.mean())

# Quick demo: run a few steps at T=2.0 (below critical temperature)
N = 30
lat = init_lattice(N, "random")
print(f"Estado inicial: magnetización = {magnetization(lat):.3f}")
for _ in range(50):
    lat = metropolis_step(lat, T=2.0)
print(f"Después de 50 barridos (T=2.0): magnetización = {magnetization(lat):.3f}")
print("\nCon T < Tc ≈ 2.27, los espines se alinean → magnetización sube")

---
## Sección 3: Visualización — Viendo la Física

Vamos a simular el modelo a diferentes temperaturas y ver cómo se ve la rejilla.

Temperaturas de interés:
- $T = 1.0$: muy frío — casi todos los espines alineados (magnético)
- $T \approx 2.27$: temperatura crítica $T_c$ — estructuras de todos los tamaños
- $T = 4.0$: muy caliente — caos aleatorio (paramagnético)

In [ ]:
def simulate(N, T, n_steps_burn=500, n_steps_measure=200):
    """
    Simula el modelo de Ising a temperatura T.
    Returns: magnetización promedio y configuración final.
    """
    lat = init_lattice(N, "random")
    # Burn-in: dejar que el sistema equilibre
    for _ in range(n_steps_burn):
        lat = metropolis_step(lat, T)
    # Medir
    mags = [magnetization(lat) for _ in range(n_steps_measure)
            if (metropolis_step(lat, T) is not None)]
    return np.mean(mags), lat

N = 40  # <-- CHANGE THIS (prueba 20, 40, 80)
temperatures = [1.0, 2.0, 2.27, 2.6, 4.0]

fig, axes = plt.subplots(1, len(temperatures), figsize=(15, 4))
cmap = ListedColormap(["#E94F37", "#2E86AB"])   # rojo=-1, azul=+1

print("Simulando... (puede tardar unos segundos)")
for ax, T in zip(axes, temperatures):
    _, lat = simulate(N, T, n_steps_burn=300)
    ax.imshow(lat, cmap=cmap, vmin=-1, vmax=1, interpolation="nearest")
    ax.set_title(f"T = {T}\nM = {magnetization(lat):.2f}", fontsize=10)
    ax.axis("off")

fig.suptitle(f"Modelo de Ising — rejilla {N}×{N}\nRojo = espín -1, Azul = espín +1",
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

### ¿Qué ves?

- **T = 1.0**: casi todo azul o casi todo rojo. El sistema está magnetizado.
- **T = 2.27** (temperatura crítica): estructuras de todos los tamaños. Esto es lo que los físicos
  llaman *criticidad* — escala invariante. Es visualmente distinto a todo lo demás.
- **T = 4.0**: mosaico aleatorio. Sin correlaciones. Completamente desordenado.

La temperatura crítica para el modelo de Ising 2D fue calculada exactamente por Lars Onsager en 1944:

$$T_c = \frac{2J}{k_B \ln(1 + \sqrt{2})} \approx 2.269 \frac{J}{k_B}$$

Es uno de los pocos resultados exactos en física estadística.

---
## Sección 4: La Transición de Fase

Lo más fascinante del modelo de Ising es la **transición de fase**: la magnetización promedio
cambia drásticamente alrededor de $T_c$.

Abajo de $T_c$: el sistema se magnetiza espontáneamente.
Arriba de $T_c$: el sistema es desordenado.

En el punto crítico exacto: la magnetización es cero pero hay correlaciones de largo alcance.

In [ ]:
# Curva de magnetización vs temperatura
# (Esto tarda 1-2 minutos — es la parte 'pesada' del notebook)
N = 25  # <-- CHANGE THIS (más grande = más preciso pero más lento)
temps = np.linspace(1.5, 3.5, 20)  # <-- CHANGE THIS (más puntos = más suave)
n_burn = 300
n_measure = 100

mags = []
print(f"Calculando magnetización para {len(temps)} temperaturas en rejilla {N}×{N}...")
print("(Tc exacta ≈ 2.269)")
for i, T in enumerate(temps):
    m, _ = simulate(N, T, n_steps_burn=n_burn, n_steps_measure=n_measure)
    mags.append(m)
    if (i+1) % 5 == 0:
        print(f"  {i+1}/{len(temps)} temperaturas completadas")

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(temps, mags, "o-", color=COLORS["blue"], lw=2, ms=6, label=f"MC ($N={N}$)")
ax.axvline(2.269, color=COLORS["red"], ls="--", lw=1.5, label="$T_c$ exacta = 2.269")
ax.fill_betweenx([0, 1], 1.5, 2.269, alpha=0.07, color=COLORS["red"], label="Fase magnética")
ax.fill_betweenx([0, 1], 2.269, 3.5, alpha=0.07, color=COLORS["blue"], label="Fase paramagnética")
ax.set_xlabel("Temperatura $T$")
ax.set_ylabel("Magnetización $\langle|M|\rangle$")
ax.set_title("Transición de fase en el modelo de Ising 2D")
ax.legend(fontsize=9)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

---
## Sección 5: Conexión con Machine Learning

El modelo de Ising no es solo un juguete de física. Es el padre intelectual de varias ideas
centrales en ML moderno.

### Máquinas de Boltzmann

Una **Restricted Boltzmann Machine (RBM)** tiene exactamente la misma estructura:
- Neuronas visibles y ocultas en lugar de espines
- Energía $H(\mathbf{v}, \mathbf{h}) = -\mathbf{v}^T W \mathbf{h} - \mathbf{b}^T \mathbf{v} - \mathbf{c}^T \mathbf{h}$
- La distribución de equilibrio es Boltzmann: $P(\mathbf{v}, \mathbf{h}) \propto e^{-H/T}$
- El entrenamiento usa Metropolis-Hastings (contrastive divergence)

Las RBMs fueron el corazón del *deep learning* de la época 2006-2012 (Hinton et al.).

### Modelos de difusión (DALL-E, Stable Diffusion)

Los modelos de difusión modernos son esencialmente:
1. Añadir ruido gradualmente a una imagen (proceso de "calentamiento" → fase desordenada)
2. Aprender a revertir ese proceso (Metropolis-Hastings en espacio continuo)

La conexión matemática es directa: ambos operan sobre la distribución de Boltzmann en espacios de alta dimensión.

### El energy landscape de redes neuronales

Los mínimos locales de la función de pérdida en redes neuronales se estudian con los mismos
herramientas del modelo de Ising. La "temperatura" en el entrenamiento es el learning rate.

In [ ]:
# 🔧 Ejercicio: encuentra la temperatura crítica empíricamente
# La magnetización debería caer a ~0 justo en Tc.
# Usa la 'susceptibilidad magnética': χ = N * (⟨M²⟩ - ⟨|M|⟩²) / T
# χ tiene un pico en Tc (diverge en el límite N→∞)

N = 20   # <-- CHANGE THIS
temps_fine = np.linspace(1.8, 3.0, 25)   # <-- CHANGE THIS

m_vals = []
m2_vals = []
print(f"Calculando susceptibilidad para rejilla {N}×{N}...")
for T in temps_fine:
    lat = init_lattice(N, "random")
    for _ in range(300):
        lat = metropolis_step(lat, T)
    ms = []
    for _ in range(100):
        lat = metropolis_step(lat, T)
        ms.append(magnetization(lat))
    ms = np.array(ms)
    m_vals.append(ms.mean())
    m2_vals.append((ms**2).mean())
    
m_vals = np.array(m_vals)
m2_vals = np.array(m2_vals)
# Susceptibility: chi = N^2 * (⟨M^2⟩ - ⟨|M|⟩^2) / T
# Note: magnetization() returns |M|/N^2, so ⟨|M|⟩^2 != ⟨M^2⟩
# simpler: chi ∝ variance of M
chi = N**2 * (m2_vals - m_vals**2) / temps_fine

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(temps_fine, m_vals, "o-", color=COLORS["blue"], ms=5)
ax1.axvline(2.269, color=COLORS["red"], ls="--", lw=1.5, label="$T_c$ = 2.269")
ax1.set_xlabel("T"); ax1.set_ylabel("$\langle|M|\rangle$"); ax1.legend()
ax1.set_title("Magnetización")

ax2.plot(temps_fine, chi, "s-", color=COLORS["orange"], ms=5)
ax2.axvline(2.269, color=COLORS["red"], ls="--", lw=1.5, label="$T_c$ = 2.269")
# Mark the peak
tc_empirical = temps_fine[np.argmax(chi)]
ax2.axvline(tc_empirical, color=COLORS["green"], ls=":", lw=1.5, 
            label=f"Pico empírico = {tc_empirical:.3f}")
ax2.set_xlabel("T"); ax2.set_ylabel("$\chi$ (susceptibilidad)")
ax2.set_title("Susceptibilidad magnética — pico en $T_c$")
ax2.legend()
plt.tight_layout()
plt.show()
print(f"Temperatura crítica empírica (pico de χ): {tc_empirical:.3f}")
print(f"Temperatura crítica exacta: 2.269")

### Reflexión final

Lo que hiciste aquí es exactamente lo que Metropolis et al. hicieron en 1953 con la ENIAC:
simular un sistema físico que no puede analizarse analíticamente, usando aleatoriedad controlada.

El mismo principio, 70 años después, corre en los servidores de OpenAI y Stability AI para generar imágenes.

La aleatoriedad no es un hack — es el motor.